# **Applied Data Science Project:** *Apartment Rent Predictor*
## Name: Aiden Zeigler

---

## Question I want to answer:
### - *"What factors are most important in determining the price of an apartment near Golden, Colorado?"*
### - *"What determines if a new apartment listing is undervalued or overvalued?"*

## **Step 1:** *Import required libraries*

In [69]:
# Supress warnings
import warnings
warnings.filterwarnings("ignore", category=Warning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# from sklearn.ensemble import RandomForestClassifier | Which model am I using?
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import GridSearchCV
import seaborn as sns

# For API
import requests
import os
import json
from pathlib import Path

## **Step 2:** *Read and load dataset*

### **SOURCE:** RentCast Rental Listings API
*Link to website:* https://developers.rentcast.io/reference/rental-listings-long-term  
API Key (required) is stored in *RENTCAST_API_KEY.txt*

---

We will set our focus on listings not in a specific city (or cities), but rather in a certain radius from the Colorado School of Mines campus. Therefore, we aren't capturing specific Golden, Wheatridge, or Lakewood, but instead any rentals that are close enough to the school.  

The Colorado School of Mines is located at *1500 Illinois St, Golden, CO 80401*, or coordinates *(39.751022, -105.222580)*

---

**NOTE:** Will revisit the possibly of web scraping rental listing websites if time allows.
Many popular rental listing websites have scrict "anti-bot" rules that prevent me from preforming my own scrapes.

In [74]:
# RentCast has subscriptions in place for those who'd like to use their APIs.
# Therefore, if we already have the data saved, we don't want to send a request.
if Path("listings.csv").is_file():
    df = pd.read_csv("listings.csv")
    
else:
    # API URL
    url = "https://api.rentcast.io/v1/listings/rental/long-term"

    # Read API Key from file
    with open("RENTCAST_API_KEY.txt") as file:
        API_KEY = file.read().strip()

    # Get a response
    response = requests.get(
        url,
        headers={
            "X-Api-Key": API_KEY,
            "Accept": "application.json"
        },
        params={
            "latitude": "39.751022",        # Latitude of Mines
            "longitude": "-105.222580",     # Longitude of Mines
            "radius": "50",                 # Search for closest listings in a 50 mile radius of Mines
            "status": "Active",             # We want active, current listings
            "limit": "500",                 # Limit to 500 listings (default)
        },
        timeout=30
    )

    # Parse JSON
    listings = response.json()

    # Define fields we want to keep
    fields = ["id", "formattedAddress", "city", "state", "zipCode",
              "price", "latitude", "longitude", "bedrooms", 
              "bathrooms", "squareFootage", "propertyType",
              "status", "listedDate", "daysOnMarket"]

    # Read to pandas DataFrame
    df = pd.DataFrame([
        {field: listing.get(field) for field in fields}
        for listing in listings
    ])
    
    # Save the data to a .csv for future use
    df.to_csv("listings.csv")
    
# Print first few rows
df.head()

,Unnamed: 0,id,formattedAddress,city,state,zipCode,price,latitude,longitude,bedrooms,bathrooms,squareFootage,propertyType,status,listedDate,daysOnMarket
0,0,"3233-Eliot-St,-Apt-211,-Denver,-CO-80211","3233 Eliot St, Apt 211, Denver, CO 80211",Denver,CO,80211,1365,39.762531,-105.024109,1,1.0,415.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
1,1,"19672-E-49th-Ave,-Denver,-CO-80249","19672 E 49th Ave, Denver, CO 80249",Denver,CO,80249,2827,39.785152,-104.760556,3,2.5,1845.0,Single Family,Active,2026-10-15T00:00:00.000Z,1
2,2,"1109-N-Marion-St,-Apt-3,-Denver,-CO-80218","1109 N Marion St, Apt 3, Denver, CO 80218",Denver,CO,80218,1070,39.734043,-104.972092,1,1.0,425.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
3,3,"2920-W-32nd-Ave,-Apt-112,-Denver,-CO-80211","2920 W 32nd Ave, Apt 112, Denver, CO 80211",Denver,CO,80211,1285,39.761635,-105.024200,1,1.0,500.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
4,4,"2920-W-32nd-Ave,-Apt-314,-Denver,-CO-80211","2920 W 32nd Ave, Apt 314, Denver, CO 80211",Denver,CO,80211,1485,39.761635,-105.024200,1,1.0,520.0,Apartment,Active,2026-10-15T00:00:00.000Z,1


## **Step 3:** Clean the Data

In [75]:
# Column types and summary statistics
info = df.info()
describe = df.describe()
print(info, describe)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        500 non-null    int64  
 1   id                500 non-null    object 
 2   formattedAddress  500 non-null    object 
 3   city              500 non-null    object 
 4   state             500 non-null    object 
 5   zipCode           500 non-null    int64  
 6   price             500 non-null    int64  
 7   latitude          500 non-null    float64
 8   longitude         500 non-null    float64
 9   bedrooms          500 non-null    int64  
 10  bathrooms         498 non-null    float64
 11  squareFootage     489 non-null    float64
 12  propertyType      500 non-null    object 
 13  status            500 non-null    object 
 14  listedDate        500 non-null    object 
 15  daysOnMarket      500 non-null    int64  
dtypes: float64(4), int64(5), object(7)
memory us

In [ ]:
# Check the data for missing values
missings = df.isna().sum()
print(f"Columns with missing values: {missings}")

# Drop any columns that contain missing values
df = df.dropna()

# Verify there are no missing values
missings = df.isna().sum().sum()
print(f"Final count of missing values: {missings}")

Columns with missing values: Unnamed: 0           0
id                   0
formattedAddress     0
city                 0
state                0
zipCode              0
price                0
latitude             0
longitude            0
bedrooms             0
bathrooms            2
squareFootage       11
propertyType         0
status               0
listedDate           0
daysOnMarket         0
dtype: int64
Final count of missing values: 0


## **Step 4:** Feature Engineering

## **Step 5:** Plot the data

## Step 5: Seaborn Heatmap

## Step 6: Create a Transformed DataFrame

## Step 7: Plot Transformed Data

## Step 8: Correlation Heatmap of Transformed Data

## Step 9: OLS Regression with statsmodels

## Step 10: Visualize Residuals and Predictions

## Step 11: Forecast Using Fitted Model

## Step 12: Plot Forecast vs Actual (Test Set)

## Step 13: Estimate Model1 with Constant and Time Trend

## Step 14: Forecast Using Model1 (Constant and Time Trend)

## Step 15: Plot Forecast vs Actual (Test Set) For Model1

## Step 16: Forecast Using Both Models (model and Model1)

## Step 17: Plot Forecast from Both Models (OLS vs Trend)

## Step 18: Calculate Monthly Growth for Each Independent Variable

## Step 19: Forecast the Independent Variables

## Step 20: Use the OLD Model to Forecast Weekly Sales

## Step 21: Un-log the Actual and Forecasted Data

## Step 22: Plot the Original and Forecasted Data (Un-logged)

# **CONCLUSION:**